# Evaluation

In [2]:
from google.colab import drive
drive.mount('/content/drive')
FOLDERNAME = "Visual_Anagrams_2024CVPR/baselines/tancik"
assert FOLDERNAME is not None, "[!] Enter the foldername."
import sys
sys.path.append('/content/drive/MyDrive/{}'.format(FOLDERNAME))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Requirements
open_clip for CLIP Scores.

In [16]:
!pip install -q open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00


## Rotate 90

In [9]:
# configs in examples from given IllusionDiffusion_View2.ipynb

# view 2
illusion_configs = [
    {
        "name": "flip.campfire.man",
        "prompts": ["an oil painting of people around a campfire", "an oil painting of an old man"],
        "views": ["identity", "flip"],
    },
    {
        "name": "jigsaw.houseplants.marilyn",
        "prompts": ["an oil painting of houseplants", "an oil painting of marilyn monroe"],
        "views": ["identity", "jigsaw"],
    },
    {
        "name": "inner.einstein.marilyn",
        "prompts": ["an oil painting of albert einstein", "an oil painting of marilyn monroe"],
        "views": ["identity", "inner_circle"],
    },
    {
        "name": "negate.landscape.houseplants",
        "prompts": ["a lithograph of a landscape", "a lithograph of houseplants"],
        "views": ["identity", "negate"],
    },
    {
        "name": "patch.lemur.kangaroo",
        "prompts": ["a pencil sketch of a lemur", "a pencil sketch of a kangaroo"],
        "views": ["identity", "patch_permute"],
    },
    {
        "name": "pixel.duck.rabbit",
        "prompts": ["a mosaic of a duck", "a mosaic of a rabbit"],
        "views": ["identity", "pixel_permute"],
    },
    {
        "name": "skew.tudor.skull",
        "prompts": ["an oil painting of a tudor portrait", "an oil painting of a skull"],
        "views": ["identity", "skew"],
    },
    {
        "name": "skew.taylor.rose",
        "prompts": ["an oil painting of a Taylor Swift", "an oil painting of a rose"],
        "views": ["identity", "skew"],
    },
]

outputs_folder = "/content/drive/MyDrive/Visual_Anagrams_2024CVPR/baselines/tancik/outputs/"

control = "rotate_90/"

outputs_folder = outputs_folder + control

# rotate 90
output_folder_names = [
    # single version
    "flip.campfire.man_2025-05-23_09-29-12",
    # complete version 25 seeds 90 rotate
    "flip.campfire.man_2025-05-26_06-49-51",
    "inner.einstein.marilyn_2025-05-26_06-56-00",
    "jigsaw.houseplants.marilyn_2025-05-26_06-52-55",
    "negate.landscape.houseplants_2025-05-26_06-59-05",
    "patch.lemur.kangaroo_2025-05-26_07-02-10",
    "pixel.duck.rabbit_2025-05-26_07-05-14",
    "skew.taylor.rose_2025-05-26_07-11-24",
    "skew.tudor.skull_2025-05-26_07-08-20",
]

In [10]:
import re, os

def get_png(folder):
  all_png_paths = []
  for subdir, _, files in os.walk(folder):
      for file in files:
          if file.lower().endswith(".png"):
              full_path = os.path.join(subdir, file)
              all_png_paths.append(full_path)
  return all_png_paths

def get_config(output_folder_names, illusion_configs):
  results = []
  for folder in output_folder_names:
      match = re.match(r"([a-z0-9.]+)_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}", folder)
      t_folder = outputs_folder + folder
      png_paths = get_png(t_folder)
      if match:
          base_name = match.group(1)
          matched_config = next((cfg for cfg in illusion_configs if cfg["name"] == base_name), None)
          results.append({
              "folder": t_folder,
              "config_name": base_name,
              "matched": matched_config is not None,
              "prompts": matched_config["prompts"] if matched_config else None,
              "views": matched_config["views"] if matched_config else None,
              "images_paths": png_paths,
          })
  return results

results_view2_90 = get_config(output_folder_names=output_folder_names, illusion_configs=illusion_configs)

# only cover View2
if results_view2_90:
  print("results_view2_90:")
  for i in results_view2_90:
    print(i)


results_view2_90:
{'folder': '/content/drive/MyDrive/Visual_Anagrams_2024CVPR/baselines/tancik/outputs/rotate_90/flip.campfire.man_2025-05-23_09-29-12', 'config_name': 'flip.campfire.man', 'matched': True, 'prompts': ['an oil painting of people around a campfire', 'an oil painting of an old man'], 'views': ['identity', 'flip'], 'images_paths': ['/content/drive/MyDrive/Visual_Anagrams_2024CVPR/baselines/tancik/outputs/rotate_90/flip.campfire.man_2025-05-23_09-29-12/2025-05-23_09-29-12_an_oil_painting_of_people_arou_to_an_oil_painting_of_an_old_man_1024.png']}
{'folder': '/content/drive/MyDrive/Visual_Anagrams_2024CVPR/baselines/tancik/outputs/rotate_90/flip.campfire.man_2025-05-26_06-49-51', 'config_name': 'flip.campfire.man', 'matched': True, 'prompts': ['an oil painting of people around a campfire', 'an oil painting of an old man'], 'views': ['identity', 'flip'], 'images_paths': ['/content/drive/MyDrive/Visual_Anagrams_2024CVPR/baselines/tancik/outputs/rotate_90/flip.campfire.man_2025

In [6]:
import torch
import open_clip
from PIL import Image
import os

def compute_clip_scores_full_matrix(image_paths, prompts, device="cuda", clip_model_name="ViT-B-32"):
    """
    Compute full CLIP similarity matrix S (N_images x N_prompts),
    and alignment/concealment scores across all.
    """
    # Load CLIP model
    model, _, preprocess = open_clip.create_model_and_transforms(
        clip_model_name, pretrained="laion2b_s34b_b79k")
    tokenizer = open_clip.get_tokenizer(clip_model_name)
    model.to(device).eval()

    # Encode prompts
    tokenized = tokenizer(prompts).to(device)
    with torch.no_grad():
        text_features = model.encode_text(tokenized)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)  # [P, D]

    # Encode images
    image_tensors = []
    image_names = []
    for img_path in image_paths:
        img = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
        image_tensors.append(img)
        image_names.append(os.path.basename(img_path))
    images = torch.cat(image_tensors, dim=0)  # [N, C, H, W]

    with torch.no_grad():
        image_features = model.encode_image(images)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)  # [N, D]

    # Compute similarity matrix S
    sim_matrix = image_features @ text_features.T  # [N, P]

    # A score: worst alignment among diagonal (assuming N == P or min(N, P))
    diag_len = min(sim_matrix.size(0), sim_matrix.size(1))
    diag_values = torch.diagonal(sim_matrix, 0)[:diag_len]
    A = diag_values.min().item()

    # C score: concealment (average of row and column softmax traces)
    tau = 0.01
    row_softmax = torch.nn.functional.softmax(sim_matrix / tau, dim=1)  # [N, P]
    col_softmax = torch.nn.functional.softmax(sim_matrix.T / tau, dim=1)  # [P, N]
    C_row = torch.trace(row_softmax) / row_softmax.size(0)
    C_col = torch.trace(col_softmax) / col_softmax.size(0)
    C = ((C_row + C_col) / 2).item()

    return {
        "A": round(A, 4),
        "C": round(C, 4),
        "S": sim_matrix.detach().cpu(),
        "image_names": image_names,
        "prompts": prompts
    }


In [11]:
print("N*N CLIP For View2 90：")
for entry in results_view2_90:
    if not entry["matched"]:
        print(f"❌ 无法匹配 config: {entry['config_name']}")
        continue

    image_paths = entry["images_paths"]
    prompts = entry["prompts"]

    N_clip_scores = compute_clip_scores_full_matrix(image_paths[:2], prompts)
    print(f"📊 {entry['config_name']}: A = {N_clip_scores['A']:.4f}, C = {N_clip_scores['C']:.4f}")


N*N CLIP For View2 90：
📊 flip.campfire.man: A = 0.2721, C = 0.2623
📊 flip.campfire.man: A = 0.2721, C = 0.2632
📊 inner.einstein.marilyn: A = 0.1530, C = 0.2500
📊 jigsaw.houseplants.marilyn: A = 0.1127, C = 0.2500
📊 negate.landscape.houseplants: A = 0.2977, C = 0.2946
📊 patch.lemur.kangaroo: A = 0.3308, C = 0.4965
📊 pixel.duck.rabbit: A = 0.3347, C = 0.2500
📊 skew.taylor.rose: A = 0.1556, C = 0.2500
📊 skew.tudor.skull: A = 0.1695, C = 0.2500


## Rotate 180

In [14]:
# configs in examples from given IllusionDiffusion_View2.ipynb

# view 2
illusion_configs = [
    {
        "name": "flip.campfire.man",
        "prompts": ["an oil painting of people around a campfire", "an oil painting of an old man"],
        "views": ["identity", "flip"],
    },
    {
        "name": "jigsaw.houseplants.marilyn",
        "prompts": ["an oil painting of houseplants", "an oil painting of marilyn monroe"],
        "views": ["identity", "jigsaw"],
    },
    {
        "name": "inner.einstein.marilyn",
        "prompts": ["an oil painting of albert einstein", "an oil painting of marilyn monroe"],
        "views": ["identity", "inner_circle"],
    },
    {
        "name": "negate.landscape.houseplants",
        "prompts": ["a lithograph of a landscape", "a lithograph of houseplants"],
        "views": ["identity", "negate"],
    },
    {
        "name": "patch.lemur.kangaroo",
        "prompts": ["a pencil sketch of a lemur", "a pencil sketch of a kangaroo"],
        "views": ["identity", "patch_permute"],
    },
    {
        "name": "pixel.duck.rabbit",
        "prompts": ["a mosaic of a duck", "a mosaic of a rabbit"],
        "views": ["identity", "pixel_permute"],
    },
    {
        "name": "skew.tudor.skull",
        "prompts": ["an oil painting of a tudor portrait", "an oil painting of a skull"],
        "views": ["identity", "skew"],
    },
    {
        "name": "skew.taylor.rose",
        "prompts": ["an oil painting of a Taylor Swift", "an oil painting of a rose"],
        "views": ["identity", "skew"],
    },
]

outputs_folder = "/content/drive/MyDrive/Visual_Anagrams_2024CVPR/baselines/tancik/outputs/"
control = "rotate_180/"

outputs_folder = outputs_folder + control
# rotate 180
output_folder_names_ = [
    # complete version 25 seeds 180 rotate
    "flip.campfire.man_2025-05-26_07-29-59",
    "inner.einstein.marilyn_2025-05-26_07-36-09",
    "jigsaw.houseplants.marilyn_2025-05-26_07-33-04",
    "negate.landscape.houseplants_2025-05-26_07-39-14",
    "patch.lemur.kangaroo_2025-05-26_07-42-19",
    "pixel.duck.rabbit_2025-05-26_07-45-24",
    "skew.taylor.rose_2025-05-26_07-51-36",
    "skew.tudor.skull_2025-05-26_07-48-31",
]

In [15]:
import re, os

def get_png(folder):
  all_png_paths = []
  for subdir, _, files in os.walk(folder):
      for file in files:
          if file.lower().endswith(".png"):
              full_path = os.path.join(subdir, file)
              all_png_paths.append(full_path)
  return all_png_paths

def get_config(output_folder_names, illusion_configs):
  results = []
  for folder in output_folder_names:
      match = re.match(r"([a-z0-9.]+)_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}", folder)
      t_folder = outputs_folder + folder
      png_paths = get_png(t_folder)
      if match:
          base_name = match.group(1)
          matched_config = next((cfg for cfg in illusion_configs if cfg["name"] == base_name), None)
          results.append({
              "folder": t_folder,
              "config_name": base_name,
              "matched": matched_config is not None,
              "prompts": matched_config["prompts"] if matched_config else None,
              "views": matched_config["views"] if matched_config else None,
              "images_paths": png_paths,
          })
  return results

results_view2_180 = get_config(output_folder_names=output_folder_names_, illusion_configs=illusion_configs)

# only cover View2
if results_view2_180:
  print("results_view2_180:")
  for i in results_view2_180:
    print(i)


results_view2_180:
{'folder': '/content/drive/MyDrive/Visual_Anagrams_2024CVPR/baselines/tancik/outputs/rotate_180/flip.campfire.man_2025-05-26_07-29-59', 'config_name': 'flip.campfire.man', 'matched': True, 'prompts': ['an oil painting of people around a campfire', 'an oil painting of an old man'], 'views': ['identity', 'flip'], 'images_paths': ['/content/drive/MyDrive/Visual_Anagrams_2024CVPR/baselines/tancik/outputs/rotate_180/flip.campfire.man_2025-05-26_07-29-59/2025-05-26_07-29-59_an_oil_painting_of_people_arou_to_an_oil_painting_of_an_old_man_1024.png']}
{'folder': '/content/drive/MyDrive/Visual_Anagrams_2024CVPR/baselines/tancik/outputs/rotate_180/inner.einstein.marilyn_2025-05-26_07-36-09', 'config_name': 'inner.einstein.marilyn', 'matched': True, 'prompts': ['an oil painting of albert einstein', 'an oil painting of marilyn monroe'], 'views': ['identity', 'inner_circle'], 'images_paths': ['/content/drive/MyDrive/Visual_Anagrams_2024CVPR/baselines/tancik/outputs/rotate_180/inne

In [16]:
import torch
import open_clip
from PIL import Image
import os

def compute_clip_scores_full_matrix(image_paths, prompts, device="cuda", clip_model_name="ViT-B-32"):
    """
    Compute full CLIP similarity matrix S (N_images x N_prompts),
    and alignment/concealment scores across all.
    """
    # Load CLIP model
    model, _, preprocess = open_clip.create_model_and_transforms(
        clip_model_name, pretrained="laion2b_s34b_b79k")
    tokenizer = open_clip.get_tokenizer(clip_model_name)
    model.to(device).eval()

    # Encode prompts
    tokenized = tokenizer(prompts).to(device)
    with torch.no_grad():
        text_features = model.encode_text(tokenized)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)  # [P, D]

    # Encode images
    image_tensors = []
    image_names = []
    for img_path in image_paths:
        img = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
        image_tensors.append(img)
        image_names.append(os.path.basename(img_path))
    images = torch.cat(image_tensors, dim=0)  # [N, C, H, W]

    with torch.no_grad():
        image_features = model.encode_image(images)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)  # [N, D]

    # Compute similarity matrix S
    sim_matrix = image_features @ text_features.T  # [N, P]

    # A score: worst alignment among diagonal (assuming N == P or min(N, P))
    diag_len = min(sim_matrix.size(0), sim_matrix.size(1))
    diag_values = torch.diagonal(sim_matrix, 0)[:diag_len]
    A = diag_values.min().item()

    # C score: concealment (average of row and column softmax traces)
    tau = 0.01
    row_softmax = torch.nn.functional.softmax(sim_matrix / tau, dim=1)  # [N, P]
    col_softmax = torch.nn.functional.softmax(sim_matrix.T / tau, dim=1)  # [P, N]
    C_row = torch.trace(row_softmax) / row_softmax.size(0)
    C_col = torch.trace(col_softmax) / col_softmax.size(0)
    C = ((C_row + C_col) / 2).item()

    return {
        "A": round(A, 4),
        "C": round(C, 4),
        "S": sim_matrix.detach().cpu(),
        "image_names": image_names,
        "prompts": prompts
    }


In [17]:
print("N*N CLIP For View2 180：")
for entry in results_view2_180:
    if not entry["matched"]:
        print(f"❌ 无法匹配 config: {entry['config_name']}")
        continue

    image_paths = entry["images_paths"]
    prompts = entry["prompts"]

    N_clip_scores = compute_clip_scores_full_matrix(image_paths[:2], prompts)
    print(f"📊 {entry['config_name']}: A = {N_clip_scores['A']:.4f}, C = {N_clip_scores['C']:.4f}")


N*N CLIP For View2 180：
📊 flip.campfire.man: A = 0.2682, C = 0.2742
📊 inner.einstein.marilyn: A = 0.1394, C = 0.2500
📊 jigsaw.houseplants.marilyn: A = 0.1070, C = 0.2500
📊 negate.landscape.houseplants: A = 0.3184, C = 0.7248
📊 patch.lemur.kangaroo: A = 0.3329, C = 0.2756
📊 pixel.duck.rabbit: A = 0.3127, C = 0.2500
📊 skew.taylor.rose: A = 0.1648, C = 0.2500
📊 skew.tudor.skull: A = 0.1853, C = 0.2500
